<a href="https://colab.research.google.com/github/Khaled-Walid1/EYOUTH-31011261203294-Library/blob/main/EYOUTH_31011261203294_Library.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Project Overview: city library's summer reading program
The goal is to combine, clean, and prepare data from three different sources `.db`, `.json`, `.html` for the final End of Summer Report.

#### The Workflow:
* **Task 1:** Query and merge all three data sources together.
* **Task 2:** Clean the merged data and fix hidden quality issues.
* **Task 3:** Check the data for community fairness and track changes using Git.


In [ ]:
# Setup & Import Libraries
import sqlite3
import pandas as pd
import json
from bs4 import BeautifulSoup

### Exploring the Library Database:
Before answering the business questions, I reviewed the available tables and how the records are related.

In [ ]:
# Connect to the database
conn = sqlite3.connect("library.db")
Data = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
Data

,name
0,members
1,books
2,checkouts


In [ ]:
# Exploring the library Database
members = pd.read_sql_query("SELECT * from members", conn)
books = pd.read_sql_query("SELECT * from books", conn)
checkouts = pd.read_sql_query("SELECT * from checkouts", conn)

# Check the size and preview each table
print(len(members))
print(members.head())
print("--------------------")
print(len(books))
print(books.head())
print("--------------------")
print(len(checkouts))
print(checkouts.head())

80
   member_id first_name last_name  grade neighborhood membership_status  \
0       1001      Salma   Ibrahim    8.0        Maadi            Active   
1       1002      Fares     Saleh    9.0        Maadi            Active   
2       1003     Bassel    Hegazy    6.0        Maadi            Active   
3       1004      Fares     Wahba    7.0        Maadi          inactive   
4       1005    Youssef     Halim    9.0        Maadi            Active   

    join_date  
0  2023-04-05  
1        None  
2  2025-04-23  
3  2024-10-09  
4  2024-05-05  
--------------------
32
   book_id                title         author
0      501      The Silver Kite  Amina Darwish
1      502       Desert Compass  Amina Darwish
2      503    The Lantern Maker   Adel Roushdy
3      504  Rooftop Astronomers   Adel Roushdy
4      505  Letters to the Nile      Aya Hafez
--------------------
391
   checkout_id  member_id  book_id checkout_date return_date
0         9263       1047      517    2024-10-21  2024-11-

In [ ]:
def run(query):
    return pd.read_sql_query(query, conn)

### Question 1: How much is each member borrowing?

The goal is to count the number of checkouts for each member, including members who have not borrowed any books.

In [ ]:
q1 = ''' SELECT
members.member_id,
members.first_name,
members.last_name,
COUNT(checkouts.checkout_id) AS checkout_count
FROM members
LEFT JOIN checkouts
ON members.member_id = checkouts.member_id
GROUP BY members.member_id, members.first_name, members.last_name
ORDER BY members.member_id;
'''
run(q1)

,member_id,first_name,last_name,checkout_count
0,1001,Salma,Ibrahim,1
1,1002,Fares,Saleh,2
2,1003,Bassel,Hegazy,9
3,1004,Fares,Wahba,0
4,1005,Youssef,Halim,3
...,...,...,...,...
75,1076,Dina,Wahba,7
76,1077,Lina,Rashad,6
77,1078,Habiba,Osman,0
78,1079,Rana,Osman,10


### Question 2: Which books match a chosen author pattern?

I searched the catalog for authors whose names starts with the letter D.

In [ ]:
q2 = '''
SELECT
book_id,
title,
author
FROM books
WHERE author LIKE 'D%'
ORDER BY author, title;
'''
print("Books whose authors start with D:")
run(q2)

Books whose authors start with D:


,book_id,title,author
0,507,Fossils and Fireflies,Dalia Serry
1,508,The Quiet Observatory,Dalia Serry
2,509,Marbles and Mirrors,Diaa Sultan
3,510,The Missing Metronome,Diaa Sultan


### Question 3: What are the most popular books?

The books are ranked by the number of times they have been checked out. The result shows the five most borrowed titles.

In [ ]:
q3 ='''
SELECT
books.title,
COUNT(checkouts.checkout_id) AS checkout_count
FROM books
LEFT JOIN checkouts
ON books.book_id = checkouts.book_id
GROUP BY books.book_id ,books.title
ORDER BY checkout_count DESC, books.title ASC
LIMIT 5;
'''
run(q3)

,title,checkout_count
0,The Silver Kite,57
1,Fossils and Fireflies,55
2,Circuits for Beginners,46
3,Kites Over Cairo,38
4,Storms and Sailboats,25


### Question 4: Who are the most active readers?

The members are ranked by the number of books they have borrowed. The result shows the ten members with the highest checkout counts.

In [ ]:
q4 = '''
SELECT
m.member_id,
m.first_name,
m.last_name,
Count(c.checkout_id) AS checkout_count
FROM members AS m
JOIN checkouts AS c
ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY checkout_count DESC, m.member_id ASC
LIMIT 10;
'''
run(q4)

,member_id,first_name,last_name,checkout_count
0,1034,Aya,Wahba,25
1,1044,Sherif,Saleh,21
2,1008,Ziad,Saleh,19
3,1010,Nour,Nabil,18
4,1027,Mostafa,Fouad,18
5,1018,Ahmed,Shafik,17
6,1024,Youssef,Hegazy,17
7,1065,Adam,Fahmy,17
8,1030,Reem,Osman,16
9,1047,Sara,Rashad,16


### Question 5: What does a neighborhood's activity look like further back in time?
Checkouts from `Shubra`, ordered newest to oldest, looking past the ten most recent.


In [ ]:
q5 = '''
SELECT
checkouts.checkout_id,
checkouts.member_id,
checkouts.book_id,
checkouts.checkout_date,
members.neighborhood
FROM checkouts
JOIN members
ON checkouts.member_id = members.member_id
WHERE members.neighborhood = 'Shubra'
ORDER BY checkouts.checkout_date DESC, checkouts.checkout_id DESC
LIMIT 100000000 OFFSET 10;
'''
print("In neighborhood Shubra:")
run(q5)

In neighborhood Shubra:


,checkout_id,member_id,book_id,checkout_date,neighborhood
0,9382,1076,507,2025-02-26,Shubra
1,9368,1079,525,2025-02-12,Shubra
2,9345,1075,507,2025-02-03,Shubra
3,9352,1076,501,2025-02-02,Shubra
4,9359,1077,501,2025-01-26,Shubra
5,9363,1079,507,2024-12-23,Shubra
6,9355,1075,515,2024-11-10,Shubra
7,9343,1075,529,2024-10-04,Shubra
8,9344,1076,507,2024-09-12,Shubra
9,9364,1077,513,2024-08-18,Shubra


## Bringing the three sources

### Stage 1: Members and Checkouts

In [ ]:
# Count how many checkouts belong to each member
checkout_counts = checkouts["member_id"].value_counts().rename("total_borrowed")

# Add the member details to each checkout
stage1 = checkouts.merge(members, on = "member_id", how = "left")

# Add the total number of checkouts for each member
stage1 = stage1.merge(checkout_counts, on = "member_id", how = "left")

stage1["total_borrowed"] = stage1["total_borrowed"].fillna(0).astype(int)

print("num of members:", len(members))
print("Checkouts before marge:", len(checkouts))
print("Checkouts after marge:", len(stage1))

print("----------------------------")
stage1.head()

num of members: 80
Checkouts before marge: 391
Checkouts after marge: 391
----------------------------


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,total_borrowed
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,16
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,14
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,5
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,6
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,10


### Stage 2: Book Details

In [ ]:
# Load the extra book information from the JSON file
with open("books.json", "r") as f:
  catalog = json.load(f)

books_json = pd.DataFrame(catalog)

# Add the book details to the checkout data
stage2 = stage1.merge(books, on = "book_id", how = "left")
stage2 = stage2.merge(books_json, on="book_id", how= "left")

print("num of books:", len(books_json))
print("stage 1 rows:", len(stage1))
print("stage 2 rows:", len(stage2))

stage2.head()

num of books: 32
stage 1 rows: 391
stage 2 rows: 391


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,total_borrowed,title,author,genre,pages,publication_year,publisher
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,16,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,14,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,5,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,6,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,10,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press


### Stage 3: Reading Kickoff Checkouts

In [ ]:
# Read the Reading Kickoff webpage
with open("summer_checkouts.html","r") as f:
  html = f.read()

soup = BeautifulSoup(html, "html.parser")

In [ ]:
# Get the checkout rows from the table
r = []

for raw in soup.select("table tr")[1:]:
  cells = [cell.get_text() for cell in raw.select("td")]
  r.append(cells)

# Create a dataframe using the same basic checkout fields
read_kickoff = pd.DataFrame(r, columns = ["member_id", "book_id", "checkout_date"])

read_kickoff.head()

,member_id,book_id,checkout_date
0,1026,522,2025-07-11
1,1049,520,2025-07-11
2,1062,525,2025-07-05
3,1065,520,2025-07-07
4,1104,515,2025-07-07


In [ ]:
# Create a dataframe using the same basic checkout fields
read_kickoff["member_id"] = read_kickoff["member_id"].astype(int)
read_kickoff["book_id"] = read_kickoff["book_id"].astype(int)

In [ ]:
# Add the member information
read_kickoff = read_kickoff.merge(members, on = "member_id", how = "left")

# Add the total number of database checkouts for each member
read_kickoff = read_kickoff.merge(checkout_counts, on ="member_id", how="left")

# Add the book information
read_kickoff = read_kickoff.merge(books, on = "book_id", how = "left")

# Add extra catalog information
read_kickoff = read_kickoff.merge(books_json, on = "book_id", how = "left")

# These fields are not provided by the Reading Kickoff page
read_kickoff["checkout_id"] = pd.NA
read_kickoff["return_date"] = pd.NA


# Members who are not in the database have no checkout count there
read_kickoff["total_borrowed"] = read_kickoff["total_borrowed"].fillna(0).astype(int)


read_kickoff.head()

,member_id,book_id,checkout_date,first_name,last_name,grade,neighborhood,membership_status,join_date,total_borrowed,title,author,genre,pages,publication_year,publisher,checkout_id,return_date
0,1026,522,2025-07-11,Nada,Saleh,7.0,Nasr City,inactive,2023-11-16,3,The Puzzle Merchant,Karim Elwy,Mystery,104,2016.0,Cairo Young Readers,<NA>,<NA>
1,1049,520,2025-07-11,Ahmed,Gamal,6.0,Heliopolis,Active,2023-06-10,1,The Copper Telescope,Jasmine Wahdan,Science Fiction,136,2009.0,Oasis Books,<NA>,<NA>
2,1062,525,2025-07-05,Tarek,Adel,7.0,Zamalek,Active,2023-01-22,7,Storms and Sailboats,Mahmoud Rafei,Adventure,297,2015.0,Oasis Books,<NA>,<NA>
3,1065,520,2025-07-07,Adam,Fahmy,6.0,Zamalek,Active,2025-07-12,17,The Copper Telescope,Jasmine Wahdan,Science Fiction,136,2009.0,Oasis Books,<NA>,<NA>
4,1104,515,2025-07-07,NaN,NaN,NaN,NaN,NaN,NaN,0,Songs of the Oasis,Hoda Bakry,Poetry,316,2022.0,Cairo Young Readers,<NA>,<NA>


In [ ]:
# Keep the same columns as the main checkout dataset
read_kickoff = read_kickoff.reindex(columns = stage2.columns)


# Add the Reading Kickoff records to the database records
combined_data = pd.concat([stage2, read_kickoff], ignore_index= True)


combined_data.head()

,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,total_borrowed,title,author,genre,pages,publication_year,publisher
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,16,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,14,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,5,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,6,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,10,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press


In [ ]:
# checks
print("stage 1 rows:", len(stage1))
print("stage 2 rows:", len(stage2))
print("read kickoff rows:", len(read_kickoff))
print("combined rows:", len(combined_data))
print("--------------------------------")

print(combined_data.columns.tolist())

print("-----------------------------------")

print(combined_data.head())

stage 1 rows: 391
stage 2 rows: 391
read kickoff rows: 26
combined rows: 417
--------------------------------
['checkout_id', 'member_id', 'book_id', 'checkout_date', 'return_date', 'first_name', 'last_name', 'grade', 'neighborhood', 'membership_status', 'join_date', 'total_borrowed', 'title', 'author', 'genre', 'pages', 'publication_year', 'publisher']
-----------------------------------
  checkout_id  member_id  book_id checkout_date return_date first_name  \
0        9263       1047      517    2024-10-21  2024-11-07       Sara   
1        9340       1072      513    2025-08-24  2025-09-01       Seif   
2        9231       1053      523    2024-02-04  2024-02-16       Adam   
3        9129       1032      513    2025-06-21  2025-06-29       Nada   
4        9370       1079      511    2025-11-11  2025-12-03       Rana   

  last_name  grade neighborhood membership_status   join_date  total_borrowed  \
0    Rashad    NaN   Heliopolis          Inactive  2024-06-25              16   
1

In [ ]:
print(combined_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 417 entries, 0 to 416
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   checkout_id        391 non-null    object 
 1   member_id          417 non-null    int64  
 2   book_id            417 non-null    int64  
 3   checkout_date      417 non-null    object 
 4   return_date        326 non-null    object 
 5   first_name         412 non-null    object 
 6   last_name          412 non-null    object 
 7   grade              376 non-null    float64
 8   neighborhood       412 non-null    object 
 9   membership_status  412 non-null    object 
 10  join_date          406 non-null    object 
 11  total_borrowed     417 non-null    int64  
 12  title              417 non-null    object 
 13  author             417 non-null    object 
 14  genre              417 non-null    object 
 15  pages              417 non-null    int64  
 16  publication_year   382 non

In [ ]:
print("stage 1 rows:", len(stage1))
print("stage 2 rows:", len(stage2))
print("read kickoff rows:", len(read_kickoff))
print("combined rows:", len(combined_data))

print("------------------------------------------")

display(combined_data.head())

stage 1 rows: 391
stage 2 rows: 391
read kickoff rows: 26
combined rows: 417
------------------------------------------


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,total_borrowed,title,author,genre,pages,publication_year,publisher
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,16,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,14,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,5,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,6,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,10,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press


In [ ]:
# Save the final combined dataset
combined_data.to_csv("task1_combined_data.csv", index = False)